# Урок 9. Практикум ЕГЭ: исполнители и анализ программ

11 класс · II четверть

[⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/index.ipynb) · [← Урок 8](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-08.ipynb) · [Урок 10 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-10.ipynb)

---

Задания на формальных исполнителей и анализ готовых программ. Обратный ход и перебор. Автоматная модель исполнителя.

In [ ]:
#@title 🚀 Шаг 1. Регистрация и подготовка урока { display-mode: "form" }
#@markdown Впиши свои данные и запусти ячейку (Shift+Enter).
#@markdown Если заводил секрет `ИНФОРМАТИКА` — оставь поля
#@markdown пустыми, они заполнятся сами.
#@markdown
#@markdown Colab спросит разрешение на доступ к аккаунту — нажми
#@markdown «Разрешить». Так проверка понимает, чья это работа.
ФИО = "" #@param {type:"string"}
#@markdown Класс — с буквой, например 11А
Класс = "" #@param {type:"string"}
#@markdown
#@markdown Адрес журнала даёт учитель. Пусто — в конце урока
#@markdown получишь квитанцию, её нужно будет отправить ему сам.
Журнал = "" #@param {type:"string"}

import importlib, urllib.request
urllib.request.urlretrieve("https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/lib/schoolinf.py", "schoolinf.py")
import schoolinf as si
importlib.reload(si)
si.start(lesson="11-09", name=ФИО, klass=Класс,
         journal=Журнал)

## Разбираемся

### Два родственных типа заданий

В ЕГЭ есть группа заданий, где нужно понять поведение чужой программы
или формального исполнителя. Внешне они разные, но решаются одними
и теми же приёмами.

**Задания на исполнителей.** Дана машина с двумя-тремя командами
и требуется найти программу, переводящую одно число в другое,
или посчитать количество таких программ.

**Задания на анализ программ.** Дан код, требуется определить,
при каких входных данных он выдаст заданный результат.

### Приём 1. Обратный ход

Главный приём для исполнителей. Вместо того чтобы идти от начального
числа вперёд, идём **от результата назад**, обращая команды.

Пример. Исполнитель умеет прибавлять 1 и умножать на 2.
Сколько программ переводят 1 в 20?

Идя вперёд, вариантов слишком много. Идя назад, из 20 можно прийти
только из 19 (обратная к «+1») или из 10 (обратная к «×2»).
Дерево сужается, и задача становится обозримой.

Обратный ход естественно превращается в динамическое программирование:
заводим таблицу «сколько программ приводит в число n» и заполняем
от начального числа к целевому.

### Приём 2. Таблица достижимости

```python
способов = {старт: 1}
for n in range(старт + 1, финиш + 1):
    итог = 0
    if n - 1 >= старт:
        итог += способов.get(n - 1, 0)      # пришли командой +1
    if n % 2 == 0:
        итог += способов.get(n // 2, 0)     # пришли командой ×2
    способов[n] = итог
```

Схема та же, что для подсчёта путей в графе: каждое число «раздаёт»
своё количество способов тем, куда из него можно попасть.

### Приём 3. Запрещённые состояния

Часто в условии есть ограничение: «программа не должна проходить
через число X». Обрабатывается одной строкой — в запрещённой точке
количество способов равно нулю.

### Приём 4. Обратная задача для анализа программ

Дана программа, известен результат — найти вход. Если диапазон
входов конечен, **перебирайте его целиком**:

```python
for x in range(0, 10000):
    if программа(x) == нужный_результат:
        print(x)
```

Это не «неспортивно», а самый разумный способ. Задание проверяет
понимание работы программы, а не умение решать уравнения в уме.

### Приём 5. Симуляция исполнителя

Когда исполнитель сложный (робот на поле, машина с регистрами),
проще всего написать его симулятор и прогнать нужные программы.

### Типичные ловушки

**Границы диапазона.** «От 1 до 20» включает и 1, и 20.

**Порядок команд.** Программа «+1, ×2» и «×2, +1» дают разные
результаты — считаются как разные программы.

**Достижимость.** Не всякое число достижимо: если команды только
«×2» и «×3», нечётные числа больше единицы недостижимы вовсе.

## Смотрим, как это работает

### Пример 1. Подсчёт программ исполнителя

> Исполнитель умеет прибавлять 1 и умножать на 2.
> Сколько программ переводят число 1 в число 20?

In [ ]:
def программ(старт, финиш):
    способов = {старт: 1}
    for n in range(старт + 1, финиш + 1):
        итог = 0
        if n - 1 >= старт:
            итог += способов.get(n - 1, 0)
        if n % 2 == 0 and n // 2 >= старт:
            итог += способов.get(n // 2, 0)
        способов[n] = итог
    return способов


таблица = программ(1, 20)
print("Сколькими способами достижимо каждое число:")
for n in sorted(таблица):
    print(f"  {n:>3}: {таблица[n]:>4}")
print(f"\nОтвет: {таблица[20]} программ")

Обратите внимание на проверку `n // 2 >= старт`: без неё программа
учла бы приход из чисел, меньших стартового, которых в задаче нет.

Такие граничные условия — главный источник ошибок в этом типе заданий.
Всегда проверяйте таблицу на маленьких числах вручную: до 4 её легко
посчитать в уме и сверить.

### Пример 2. Исполнитель с запретом

> То же самое, но программа не должна проходить через число 13.

In [ ]:
def программ_с_запретом(старт, финиш, запрещённые):
    способов = {старт: 1}
    for n in range(старт + 1, финиш + 1):
        if n in запрещённые:
            способов[n] = 0
            continue
        итог = 0
        if n - 1 >= старт:
            итог += способов.get(n - 1, 0)
        if n % 2 == 0 and n // 2 >= старт:
            итог += способов.get(n // 2, 0)
        способов[n] = итог
    return способов[финиш]


print(f"Без запрета:        {программ(1, 20)[20]}")
print(f"Запрещено 13:       {программ_с_запретом(1, 20, {13})}")
print(f"Запрещены 13 и 17:  {программ_с_запретом(1, 20, {13, 17})}")

Запрет обработан двумя строками, и вся остальная логика не изменилась.
Это признак хорошо выбранного подхода: усложнение условия не требует
переписывать решение.

### Пример 3. Анализ программы обратным перебором

> Программа получает натуральное число и выполняет преобразование.
> При каких входных значениях от 1 до 1000 результат равен 12?

In [ ]:
def преобразование(x):
    результат = 0
    while x > 0:
        цифра = x % 10
        if цифра % 2 == 0:
            результат += цифра
        x //= 10
    return результат


подходящие = [x for x in range(1, 1001) if преобразование(x) == 12]

print(f"Найдено {len(подходящие)} чисел")
print(f"Первые десять: {подходящие[:10]}")
print(f"Наименьшее: {min(подходящие)}, наибольшее: {max(подходящие)}")

Разбираться в том, что именно делает программа, не потребовалось:
перебор ответил на вопрос за долю секунды.

Но понять программу всё же полезно — она суммирует чётные цифры числа.
Знание этого позволяет проверить ответ: наименьшее число с суммой
чётных цифр 12 — это 48.

## Пробуем сами

### Задача 1. Количество программ

Исполнитель умеет прибавлять 1 и умножать на 2. Верните количество
программ, переводящих `старт` в `финиш`.

In [ ]:
def количество_программ(старт, финиш):
    return ...

In [ ]:
si.check("1", количество_программ, [
    ((1, 1), 1),
    ((1, 2), 2),
    ((1, 3), 2),
    ((1, 4), 4),
    ((1, 20), 60),
])

### Задача 2. Программы с запретом

То же, но программа не должна проходить через числа из списка
запрещённых.

In [ ]:
def программ_с_запретами(старт, финиш, запрещённые):
    return ...

In [ ]:
si.check("2", программ_с_запретами, [
    ((1, 20, []), 60),
    ((1, 20, [13]), 40),
    ((1, 4, [2]), 0),
    ((1, 10, [2, 3]), 0),
])

### Задача 3. Анализ программы

Дана программа: она складывает **нечётные** цифры числа.
Верните список всех чисел от 1 до `предела`, для которых результат
равен заданному значению.

In [ ]:
def подходящие_числа(предел, результат):
    return ...

In [ ]:
si.check("3", подходящие_числа, [
    ((50, 9), [9, 29, 49]),
    ((20, 1), [1, 10, 12, 14, 16, 18]),
    ((10, 100), []),
])

## Домашнее задание

### Домашнее задание 1. Другой набор команд

Исполнитель умеет прибавлять 2 и умножать на 3.
Верните количество программ, переводящих `старт` в `финиш`.

Внимательно с условиями: прийти умножением можно только в число,
делящееся на 3.

In [ ]:
def программ_2и3(старт, финиш):
    return ...

In [ ]:
si.check("дз1", программ_2и3, [
    ((1, 1), 1),
    ((1, 3), 2),
    ((1, 9), 4),
    ((1, 12), 0),
])

### Домашнее задание 2. Через обязательную точку

Верните количество программ, переводящих `старт` в `финиш`
**обязательно через** число `через`.

Приём: количество путей через точку равно произведению количества
путей до неё на количество путей после.

Команды прежние: прибавить 1, умножить на 2.

In [ ]:
def через_точку(старт, через, финиш):
    return ...

In [ ]:
si.check("дз2", через_точку, [
    ((1, 10, 20), 28),
    ((1, 2, 4), 4),
    ((1, 5, 5), 4),
])

### Домашнее задание 3. Симулятор исполнителя

Напишите функцию, которая выполняет программу исполнителя
и возвращает итоговое число.

Программа задаётся строкой из команд:
* `"+"` — прибавить 1;
* `"*"` — умножить на 2;
* `"-"` — вычесть 1.

`выполнить(1, "++*")` → `6`, потому что 1 → 2 → 3 → 6.

In [ ]:
def выполнить(начало, программа):
    return ...

In [ ]:
si.check("дз3", выполнить, [
    ((1, "++*"), 6),
    ((1, "*+*"), 6),
    ((5, ""), 5),
    ((10, "--"), 8),
    ((1, "****"), 16),
])

---

### Проверьте себя на настоящем задании

Возьмите условие из любого варианта ЕГЭ на исполнителя и решите его
двумя способами: рассуждением на бумаге и таблицей в коде.
Ответы должны совпасть.

Если не совпали — почти наверняка дело в границах диапазона
или в забытом запрете. Это самые частые ошибки, и находить их
сравнением двух решений быстрее всего.

---

## Отчёт по уроку

Запусти ячейку ниже, когда решишь задачи. Результат уйдёт учителю автоматически.

In [ ]:
si.report()

---

[← Урок 8](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-08.ipynb) · [⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/index.ipynb) · [🏠 Ко всем классам](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/index.ipynb) · [Урок 10 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-10.ipynb)